In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


# IMPORTING IMPORTANT LIBRARIES 

In [6]:
import numpy as np
import pandas as pd

# Preprocessing & Pipelines
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

# Evaluation Metrics
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

# Tree-Based Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')




In [7]:
train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
train

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


# EDA

In [11]:
train.columns
print("-"*50)
train.info()

train['Churn'].head()


--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract         

0     No
1     No
2     No
3    Yes
4    Yes
Name: Churn, dtype: object

Okay this give us general idea we need as the dataset was already cleaned and we only need to convert Churn into number Like Yes into 0 and No into 1
Now we will mive toward the Machine Learning part

# Machine Learning 

In [12]:
y=train['Churn'].map({"Yes":1,"No":0})
X=train.drop(columns=['id','Churn'])

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Initialization complete!")
print(f"Target (y) sample:\n{y.head()}\n")
print(f"Features (X) shape: {X.shape}")


Initialization complete!
Target (y) sample:
0    0
1    0
2    0
3    1
4    1
Name: Churn, dtype: int64

Features (X) shape: (594194, 19)


In [13]:
# Split the data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")


Training features shape: (475355, 19)
Validation features shape: (118839, 19)


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 2 Bundling preprocessor and model
pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='auc'))
])
# 3 Train the model
print("Training XGBoost model... (this might take a few seconds)")
pipeline_xgb.fit(X_train, y_train)
print("Training complete!")


Training XGBoost model... (this might take a few seconds)
Training complete!


In [17]:
y_val_pred_proba = pipeline_xgb.predict_proba(X_val)[:, 1]

# 2. Calculate the ROC AUC score
score = roc_auc_score(y_val, y_val_pred_proba)

print("--- Local Validation Results ---")
print(f"XGBoost Classifier ROC AUC Score: {score}")


--- Local Validation Results ---
XGBoost Classifier ROC AUC Score: 0.9136293700115462


#LGBM 
ok this is my first time using this model as i am creating this model on just basis of tutorial vedios 

# LGBM MODEL
Okay,So this is my first time using this model i am just creating these models on just basis of two or three tutorial vedios and later on just learned their syntax is somewhat same but math behind them is different 

In [19]:

pipeline_lgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=-1))
])

print("Training LightGBM model")
pipeline_lgb.fit(X_train, y_train)

y_val_pred_lgb = pipeline_lgb.predict_proba(X_val)[:, 1]
lgb_score = roc_auc_score(y_val, y_val_pred_lgb)

print("LightGBM Validation Results")
print(f"LightGBM ROC AUC Score: {lgb_score:.5f}")


Training LightGBM model...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- LightGBM Validation Results ---
LightGBM ROC AUC Score: 0.91367


# CatBoost
Similar to LGBM Model 

In [20]:
pipeline_cat = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', CatBoostClassifier(iterations=100, learning_rate=0.05, depth=5, random_state=42, verbose=0))
])

print("Training CatBoost model")
pipeline_cat.fit(X_train, y_train)

# 3. Evaluate
y_val_pred_cat = pipeline_cat.predict_proba(X_val)[:, 1]
cat_score = roc_auc_score(y_val, y_val_pred_cat)

print("CatBoost Validation Results")
print(f"CatBoost ROC AUC Score: {cat_score:.5f}")


Training CatBoost model
CatBoost Validation Results
CatBoost ROC AUC Score: 0.91285


# FINAL RESULT

Now we will calculate Average of three models result...

In [21]:
blended_val_preds = (y_val_pred_proba + y_val_pred_lgb + y_val_pred_cat) / 3
blend_score = roc_auc_score(y_val, blended_val_preds)

print("Ensemble Blend Validation Results.")
print(f"Blended Ensemble ROC AUC Score: {blend_score}")


Ensemble Blend Validation Results.
Blended Ensemble ROC AUC Score: 0.9136089366509244


In [ ]:

X_test = test.drop(columns=['id'])

test_preds_xgb = pipeline_xgb.predict_proba(X_test)[:, 1]
test_preds_lgb = pipeline_lgb.predict_proba(X_test)[:, 1]
test_preds_cat = pipeline_cat.predict_proba(X_test)[:, 1]

sub_lgb = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')
sub_lgb['Churn'] = test_preds_lgb
sub_lgb.to_csv('submission_lightgbm.csv', index=False)

sub_blend = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')
sub_blend['Churn'] = (test_preds_xgb + test_preds_lgb + test_preds_cat) / 3
sub_blend.to_csv('submission_ensemble_blend.csv', index=False)

print("Both submission files created successfully!")
print("1. submission_lightgbm.csv")
print("2. submission_ensemble_blend.csv")
